# App Usage Dataset 
## 1. Load & Explore the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

mydata = pd.read_csv('appusage.csv')

In [ ]:
mydata.head()

In [ ]:
mydata.tail(5)

In [ ]:
mydata.info()

In [ ]:
mydata.describe()

## 2. Histograms

In [ ]:
mydata.hist(figsize=(20, 15))
plt.suptitle('Histograms of All Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
mydata.hist(by='high_productivity', column='total_screen_time')

In [ ]:
mydata.hist(by='high_productivity', column='social')

In [ ]:
mydata.hist(by='high_productivity', column='productivity')

## 3. Cross-tabulation & Pivot Tables

In [ ]:
pd.crosstab(mydata['high_productivity'], mydata['education'].apply(lambda x: 'high' if x > 0.15 else 'low'))

In [ ]:
mydata['edu_group'] = mydata['education'].apply(lambda x: 'high' if x > 0.15 else 'low')
pd.pivot_table(mydata, index=['high_productivity', 'edu_group'],
              columns=[], aggfunc=len)

In [ ]:
pd.pivot_table(mydata, 'total_screen_time',
              index=['high_productivity'],
              columns=['edu_group'])

## 4. Box Plots & Outlier Detection

In [ ]:
sns.boxplot(x='high_productivity', y='total_screen_time', data=mydata)
plt.title('Screen Time by Productivity Level')
plt.show()

In [ ]:
q1, q3 = np.percentile(mydata['total_screen_time'], [25, 75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = mydata[(mydata['total_screen_time'] < lower) | (mydata['total_screen_time'] > upper)]
print(f'Q1: {q1:.4f}, Q3: {q3:.4f}, IQR: {iqr:.4f}')
print(f'Lower bound: {lower:.4f}, Upper bound: {upper:.4f}')
print(f'Number of outliers: {len(outliers)}')

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(mydata['total_screen_time'], vert=False, patch_artist=True,
            boxprops=dict(facecolor='lightgreen', color='black'),
            flierprops=dict(marker='o', markerfacecolor='red', markersize=8))
plt.title('Box Plot: total_screen_time')
plt.xlabel('Value')
plt.show()

## 5. Count Plots

In [ ]:
mydata['social_group'] = mydata['social'].apply(lambda x: 'high' if x > 0.13 else 'low')
sns.countplot(x='high_productivity', hue='social_group', data=mydata)
plt.title('High Productivity by Social Usage Group')
plt.show()

## 6. Measures of Central Tendency & Dispersion

In [ ]:
mean_val = np.mean(mydata['total_screen_time'])
std_val  = np.std(mydata['total_screen_time'])
print(f'Mean total_screen_time : {mean_val:.4f}')
print(f'Std  total_screen_time : {std_val:.4f}')

In [ ]:
q1, q2, q3 = np.percentile(mydata['total_screen_time'], [25, 50, 75])
print(f'Q1: {q1:.4f}  Median (Q2): {q2:.4f}  Q3: {q3:.4f}')

In [ ]:
print(mydata.mean(numeric_only=True))

In [ ]:
print(mydata.std(numeric_only=True))

## 7. Skewness & Distribution Shapes

In [ ]:
from scipy import stats as st

normal_dist = np.random.normal(0.52, 0.29, 500)
skewed_dist = st.skewnorm.rvs(a=10, loc=0.52, scale=0.29, size=500)

plt.figure(figsize=(10, 5))
plt.hist(normal_dist, bins=30, alpha=0.5, label='Normal', color='skyblue')
plt.hist(skewed_dist, bins=30, alpha=0.5, label='Right Skewed', color='salmon')
plt.title('Normal vs Right-Skewed Distribution')
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
plt.axvline(x=mydata['total_screen_time'].mean(), color='orange', label='Mean')
plt.axvline(x=np.median(mydata['total_screen_time']), color='green', label='Median')
plt.hist(mydata['total_screen_time'], color='lightgray', bins=30)
plt.title('total_screen_time: Mean vs Median')
plt.legend()
plt.show()

## 8. Z-Scores & p-Values

In [ ]:
z_scores = st.zscore(mydata['total_screen_time'])
print('First 10 z-scores:', z_scores[:10])

In [ ]:
z_input = 1.5
x_val = mean_val + z_input * std_val
print(f'Raw score for Z={z_input}: {x_val:.4f}')

In [ ]:
prob = st.norm.cdf(0.52, loc=mean_val, scale=std_val)
print(f'P(total_screen_time < 0.52): {prob:.4f}')

p_below = st.norm.cdf(-2.5)
p_above = 1 - st.norm.cdf(2.5)
print(f'P-value (|Z| > 2.5): {p_below + p_above:.4f}')

## 9. Chebyshev's Rule

In [ ]:
def chebyshev_rule(k):
    if k <= 1:
        return 'k must be greater than 1'
    return (1 - (1 / k**2)) * 100

print(f'At least {chebyshev_rule(2):.2f}% within 2 SD')
print(f'At least {chebyshev_rule(3):.2f}% within 3 SD')

within_k2 = mydata[(mydata['total_screen_time'] >= mean_val - 2*std_val) &
                   (mydata['total_screen_time'] <= mean_val + 2*std_val)]
print(f'Actual within 2 SD: {100*len(within_k2)/len(mydata):.2f}%')

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(mydata['total_screen_time'], bins=30, color='gray', alpha=0.4)
plt.axvline(mean_val, color='blue', linestyle='dashed', label='Mean')
plt.axvline(mean_val - 2*std_val, color='green', linestyle='--', label='k=2 Lower')
plt.axvline(mean_val + 2*std_val, color='green', linestyle='--', label='k=2 Upper')
plt.title("Chebyshev's Theorem – total_screen_time")
plt.legend()
plt.show()

## 10. t-Tests

In [ ]:
null_mean = 0.52
t_stat, p_val = st.ttest_1samp(mydata['total_screen_time'], null_mean)
print(f'T-statistic: {t_stat:.4f}')
print(f'P-value    : {p_val:.4f}')
alpha = 0.05
if p_val < alpha:
    print('Decision: Reject H0 – mean differs significantly from', null_mean)
else:
    print('Decision: Fail to reject H0')

In [ ]:
high_prod  = mydata[mydata['high_productivity'] == 1]['total_screen_time'].values
low_prod   = mydata[mydata['high_productivity'] == 0]['total_screen_time'].values
t2, p2 = st.ttest_ind(high_prod, low_prod, equal_var=False)
print(f'High productivity mean: {high_prod.mean():.4f}')
print(f'Low  productivity mean: {low_prod.mean():.4f}')
print(f'T-statistic: {t2:.4f},  P-value: {p2:.4f}')

## 11. Confidence Intervals

In [ ]:
n    = mydata['total_screen_time'].size
s    = mydata['total_screen_time'].std()
xbar = mydata['total_screen_time'].mean()
z    = 1.96

CI_err = z * (s / n**0.5)
print(f'95% CI: ({xbar - CI_err:.4f}, {xbar + CI_err:.4f})')

fig, ax = plt.subplots()
plt.ylabel('total_screen_time')
plt.grid(axis='y')
ax.errorbar(['total_screen_time'], [xbar], [CI_err], fmt='o', color='green')
plt.title('95% Confidence Interval')
plt.show()

## 12. Correlation Analysis

In [ ]:
corr = mydata.corr(numeric_only=True)
corr

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
r, p = st.pearsonr(mydata['social'], mydata['total_screen_time'])
print(f'Pearson r = {r:.4f},  p-value = {p:.4e}')
print(f'R-squared = {r**2:.4f}')

## 13. Scatter Plots & Pair Plot

In [ ]:
sns.scatterplot(x='social', y='total_screen_time', hue='high_productivity', data=mydata)
plt.title('Social Usage vs Total Screen Time')
plt.show()

In [ ]:
sns.pairplot(mydata[['education','social','productivity','entertainment','health','total_screen_time','high_productivity']], hue='high_productivity')
plt.show()

## 14. Distribution Plots

In [ ]:
sns.displot(mydata['total_screen_time'], kde=True)
plt.title('Distribution of total_screen_time')
plt.show()

In [ ]:
for col in ['education', 'social', 'productivity', 'entertainment', 'health']:
    sns.displot(mydata[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.show()

## 15. Linear Regression (OLS)

In [ ]:
x = mydata['social'].values
y = mydata['total_screen_time'].values

cov_mat = np.cov(x, y)
beta1 = cov_mat[0, 1] / cov_mat[0, 0]
beta0 = y.mean() - beta1 * x.mean()
print(f'Intercept (β0): {beta0:.4f}')
print(f'Slope     (β1): {beta1:.4f}')

In [ ]:
xline = np.linspace(x.min(), x.max(), 1000)
yline = beta0 + beta1 * xline

sns.scatterplot(x=x, y=y, alpha=0.5)
plt.plot(xline, yline, color='orange')
plt.xlabel('social usage')
plt.ylabel('total_screen_time')
plt.title('Linear Regression: Social → Screen Time')
plt.show()

In [ ]:
import statsmodels.api as sm

X_ols = sm.add_constant(mydata[['social', 'productivity', 'entertainment', 'education', 'health']])
model = sm.OLS(mydata['total_screen_time'], X_ols)
result = model.fit()
print(result.summary())